<a href="https://colab.research.google.com/github/marcouras/AI-engineering-fundamentals/blob/main/lezione3/Lezione3_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

# 🤖 AI Engineering Fundamentals
## Lezione 3 — Conversazione & Memoria

**ITS Novitas 4.0 — Sviluppatore Intelligenza Artificiale**  
Docente: Marco Uras | 📅 Martedì 26/05/2026

---

### 🎯 Obiettivi
- ✅ Gestire la conversation history (multi-turno)
- ✅ Implementare troncamento e sliding window
- ✅ Aggiungere lo streaming delle risposte
- ✅ Salvare e ricaricare la memoria su file JSON

In [1]:
# Setup
!pip install anthropic -q
from google.colab import userdata
import anthropic, os, json

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
client = anthropic.Anthropic()
print("✅ Setup completato!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.7/838.7 kB 9.8 MB/s eta 0:00:00
✅ Setup completato!


---
## 1. Conversation History — il chatbot che ricorda

Il modello **non ha memoria propria**. Per farlo 'ricordare', dobbiamo inviargli tutta la conversazione ad ogni chiamata.

In [2]:
# Chatbot multi-turno base
history = []

def chat(messaggio, system=None):
    """Invia un messaggio mantenendo la history."""
    # 1. Aggiungi il messaggio dell'utente
    history.append({"role": "user", "content": messaggio})

    # 2. Invia TUTTA la history al modello
    params = {
        "model": "claude-haiku-4-5-20251001",
        "max_tokens": 500,
        "messages": history
    }
    if system:
        params["system"] = system

    risposta = client.messages.create(**params)
    testo = risposta.content[0].text

    # 3. Aggiungi la risposta alla history
    history.append({"role": "assistant", "content": testo})

    return testo

print("✅ Funzione chat pronta!")

✅ Funzione chat pronta!


In [3]:
# Proviamo la memoria!
history = []  # Reset

print("👤 Mi chiamo Marco e sono di Sassari.")
r1 = chat("Mi chiamo Marco e sono di Sassari.")
print(f"🤖 {r1}\n")

print("👤 Qual è la capitale della Sardegna?")
r2 = chat("Qual è la capitale della Sardegna?")
print(f"🤖 {r2}\n")

print("👤 Come mi chiamo?")
r3 = chat("Come mi chiamo?")  # Ricorda il nome?
print(f"🤖 {r3}\n")

print(f"📊 Messaggi in history: {len(history)}")

👤 Mi chiamo Marco e sono di Sassari.
🤖 Ciao Marco! Piacere di conoscerti! 👋

Sassari è una bellissima città in Sardegna, con una storia ricca e affascinante. Sei nato e cresciuto lì, o ci sei arrivato da poco?

C'è qualcosa in cui posso aiutarti?

👤 Qual è la capitale della Sardegna?
🤖 La capitale della Sardegna è **Cagliari**. 🏛️

Si trova nella parte meridionale dell'isola ed è la città più grande della regione. Sassari, dove sei tu, è invece il secondo centro più importante dell'isola.

👤 Come mi chiamo?
🤖 Ti chiami **Marco**! 😊

Me l'hai detto all'inizio della nostra conversazione.

📊 Messaggi in history: 6


In [4]:
# Vediamo quanti token stiamo usando
from anthropic import Anthropic

count = client.messages.count_tokens(
    model="claude-haiku-4-5-20251001",
    messages=history
)
print(f"📊 Token nella history attuale: {count.input_tokens}")
print(f"💰 Costo stimato prossima chiamata: ${count.input_tokens / 1_000_000 * 1.0:.6f}")
print()
print("💡 Nota: la history cresce ad ogni messaggio — dobbiamo gestirla!")

📊 Token nella history attuale: 214
💰 Costo stimato prossima chiamata: $0.000214

💡 Nota: la history cresce ad ogni messaggio — dobbiamo gestirla!


---
## 2. Gestire la Context Window

Tre strategie per evitare che la history cresca all'infinito.

In [5]:
# STRATEGIA 1: Truncation
MAX_MESSAGGI = 6  # massimo 3 turni (user + assistant)

def chat_con_troncamento(messaggio, system=None):
    history.append({"role": "user", "content": messaggio})

    # Tronca se troppo lunga
    if len(history) > MAX_MESSAGGI:
        history[:] = history[-MAX_MESSAGGI:]
        print(f"  ✂️  History troncata a {MAX_MESSAGGI} messaggi")

    risposta = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=300,
        messages=history
    )
    testo = risposta.content[0].text
    history.append({"role": "assistant", "content": testo})
    return testo

# Test
history = []
for i in range(5):
    r = chat_con_troncamento(f"Messaggio numero {i+1}. Quanti messaggi ricordi?")
    print(f"Turno {i+1} | History: {len(history)} msg | Risposta: {r[:80]}...")

Turno 1 | History: 2 msg | Risposta: Ricordo **1 messaggio** - questo che hai appena scritto.

In questa conversazion...
Turno 2 | History: 4 msg | Risposta: Ricordo **2 messaggi**:

1. "Messaggio numero 1. Quanti messaggi ricordi?"
2. "M...
Turno 3 | History: 6 msg | Risposta: Ricordo **3 messaggi**:

1. "Messaggio numero 1. Quanti messaggi ricordi?"
2. "M...
  ✂️  History troncata a 6 messaggi
Turno 4 | History: 7 msg | Risposta: Ricordo **4 messaggi**:

1. "Messaggio numero 1. Quanti messaggi ricordi?"
2. "M...
  ✂️  History troncata a 6 messaggi
Turno 5 | History: 7 msg | Risposta: Ricordo **5 messaggi**:

1. "Messaggio numero 1. Quanti messaggi ricordi?"
2. "M...


In [6]:
# STREAMING — output token per token
print("🌊 Risposta in streaming:\n")

full_text = ""
with client.messages.stream(
    model="claude-haiku-4-5-20251001",
    max_tokens=300,
    messages=[{"role": "user", "content": "Raccontami in 3 frasi cos'è il machine learning."}]
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)
        full_text += text

print(f"\n\n📊 Testo totale: {len(full_text)} caratteri")

🌊 Risposta in streaming:

# Machine Learning

Il machine learning è una branca dell'intelligenza artificiale che permette ai computer di imparare dai dati senza essere esplicitamente programmati per ogni situazione. Gli algoritmi analizzano grandi quantità di informazioni per identificare pattern e regole, migliorando le loro prestazioni man mano che elaborano più dati. In pratica, invece di seguire istruzioni fisse, questi sistemi sviluppano autonomamente la capacità di prendere decisioni e fare previsioni basandosi sull'esperienza.

📊 Testo totale: 513 caratteri


---
## 3. Memoria Persistente su JSON

Salviamo la history su file — il chatbot ricorda tra sessioni diverse.

In [7]:
import json, os

MEMORY_FILE = "chat_history.json"

def carica_storia():
    """Carica la history dal file JSON. Restituisce lista vuota se non esiste."""
    if os.path.exists(MEMORY_FILE):
        with open(MEMORY_FILE, "r", encoding="utf-8") as f:
            storia = json.load(f)
            print(f"📂 Storia caricata: {len(storia)} messaggi precedenti")
            return storia
    print("🆕 Nessuna storia precedente — nuova conversazione")
    return []

def salva_storia(history):
    """Salva la history su file JSON."""
    with open(MEMORY_FILE, "w", encoding="utf-8") as f:
        json.dump(history, f, ensure_ascii=False, indent=2)
    print(f"💾 Storia salvata: {len(history)} messaggi")

def chat_persistente(messaggio, system=None):
    """Chatbot con memoria persistente tra sessioni."""
    history.append({"role": "user", "content": messaggio})

    params = {
        "model": "claude-haiku-4-5-20251001",
        "max_tokens": 400,
        "messages": history
    }
    if system:
        params["system"] = system

    risposta = client.messages.create(**params)
    testo = risposta.content[0].text
    history.append({"role": "assistant", "content": testo})

    salva_storia(history)  # Salva dopo ogni messaggio
    return testo

print("✅ Funzioni di persistenza pronte!")

✅ Funzioni di persistenza pronte!


In [8]:
# Simulazione sessione 1
print("=" * 50)
print("SESSIONE 1")
print("=" * 50)

history = carica_storia()  # Carica eventuali messaggi precedenti

r = chat_persistente("Ciao! Mi chiamo Luca e studio AI engineering a Sassari.")
print(f"🤖 {r}\n")

r = chat_persistente("Qual è la lezione più difficile secondo te?")
print(f"🤖 {r}\n")

print("\n--- Fine sessione 1 ---")
print(f"History salvata con {len(history)} messaggi")

SESSIONE 1
🆕 Nessuna storia precedente — nuova conversazione
💾 Storia salvata: 2 messaggi
🤖 Ciao Luca! Piacere di conoscerti! 👋

Che bello studiare AI engineering a Sassari! È un campo affascinante e in grande crescita. Stai seguendo un percorso di laurea triennale o magistrale?

Se vuoi, posso aiutarti con:
- Dubbi su concetti di machine learning, deep learning, ecc.
- Progetti di programmazione
- Domande su tecnologie e framework (Python, TensorFlow, PyTorch...)
- O semplicemente chiacchierare del campo

Come procede lo studio? Ci sono argomenti che ti interessano particolarmente o su cui stai lavorando adesso?

💾 Storia salvata: 4 messaggi
🤖 Buona domanda! Secondo la mia esperienza nel rispondere a studenti, direi che **i concetti più ostici sono**:

1. **Le matematiche sottostanti** - Algebra lineare, calcolo multivariato, probabilità. Molti vedono formule senza capire il "perché" dietro. È facile imparare a usare una libreria, difficile capire cosa accade dentro.

2. **Il salto tr

In [9]:
# Simulazione sessione 2 — ricarica la memoria!
print("=" * 50)
print("SESSIONE 2 (nuova sessione, stessa memoria)")
print("=" * 50)

history = carica_storia()  # Ricarica dal file

r = chat_persistente("Come mi chiamo? Ricordi cosa stavo studiando?")
print(f"🤖 {r}")

SESSIONE 2 (nuova sessione, stessa memoria)
📂 Storia caricata: 4 messaggi precedenti
💾 Storia salvata: 6 messaggi
🤖 Sì, certo! 😊

Ti chiami **Luca** e studi **AI engineering a Sassari**.

L'ho ricordato dalla nostra conversazione all'inizio - non è che ho una memoria magica, semplicemente ho letto quello che mi hai scritto poco fa! In questa conversazione posso tenere traccia di quello che mi dici.

Però va bene sottolinearlo: non ricordo le nostre chat precedenti (se le abbiamo avute). Ogni nuova conversazione parte da zero per me, a meno che tu non mi ricordi i dettagli.

C'è qualcosa di specifico del tuo studio su cui vuoi parlare? 👨‍💻


---
## ⭐ Esercizi

In [10]:
NOME_STUDENTE = "Lorenzo"  # ← SCRIVI IL TUO NOME
if NOME_STUDENTE:
    print(f"✅ Notebook di: {NOME_STUDENTE}")
else:
    print("⚠️ Scrivi il tuo nome!")

✅ Notebook di: Lorenzo


### Esercizio 1 — Chatbot multi-turno base ★☆☆
Crea un chatbot con system prompt WiData che mantiene la history. Fai almeno 4 domande collegate e verifica che risponda in modo coerente con i messaggi precedenti.

In [11]:
# ESERCIZIO 1
history = []
system_widata = """ # ← scrivi il system prompt WiData
Sei un assistente di supporto clienti WiData.
Rispondi in modo chiaro, professionale e coerente con la conversazione.
Mantieni il contesto dei messaggi precedenti.
"""

# Fai almeno 4 domande collegate
print("👤 Quali prodotti IoT offrite?")
print("🤖", chat("Quali prodotti IoT offrite?", system=system_widata))

print("\n👤 Mi puoi spiegare meglio il monitoraggio ambientale?")
print("🤖", chat("Mi puoi spiegare meglio il monitoraggio ambientale?"))

print("\n👤 E per quale settore è più utile?")
print("🤖", chat("E per quale settore è più utile?"))

print("\n👤 Riassumimi tutto in una frase.")
print("🤖", chat("Riassumimi tutto in una frase."))

👤 Quali prodotti IoT offrite?
🤖 # Prodotti IoT WiData

Purtroppo non ho accesso a un catalogo dettagliato e aggiornato dei nostri prodotti IoT al momento. Per fornirvi informazioni accurate e complete, vi consiglio di:

## Contattarci direttamente:
- **Sito web:** www.widata.it
- **Email:** support@widata.it
- **Telefono:** consulta il sito per i numeri di contatto

## Cosa posso aiutarvi a fare:
- Rispondere a domande generiche sui servizi IoT
- Assistere con problemi tecnici
- Fornire supporto sui vostri prodotti attuali

**Avete un prodotto WiData specifico di cui avete bisogno di supporto?** Sarò felice di aiutarvi con dettagli tecnici o problematiche di utilizzo! 🚀

👤 Mi puoi spiegare meglio il monitoraggio ambientale?
🤖 # Monitoraggio Ambientale IoT

Sono felice di spiegarvi questo tema! Tuttavia, devo essere trasparente: **non ho accesso a documentazione specifica sui nostri servizi di monitoraggio ambientale**.

## Cosa generalmente include il monitoraggio ambientale IoT:

**Se

### Esercizio 2 — Sliding Window ★★☆
Implementa la sliding window: mantieni sempre il system prompt + gli ultimi 4 turni (8 messaggi). Testa che dopo 6 turni il chatbot non ricordi più i primi messaggi ma ricordi gli ultimi.

In [12]:
# ESERCIZIO 2
MAX_TURNS = 4 # mantieni solo gli ultimi 4 turni

def chat_sliding_window(messaggio, system=None):
 history.append({"role": "user", "content": messaggio})

 # TODO: implementa sliding window
 # Suggerimento: history[-MAX_TURNS*2:] mantiene gli ultimi N turni
 if len(history) > MAX_TURNS * 2:
  history[:] = history[-MAX_TURNS * 2:]

 params = {
  "model": "claude-haiku-4-5-20251001",
  "max_tokens": 300,
  "messages": history
 }
 if system:
  params["system"] = system

 risposta = client.messages.create(**params)
 testo = risposta.content[0].text
 history.append({"role": "assistant", "content": testo})
 return testo

# Test: la prima informazione viene dimenticata dopo 4 turni?
history = []
print("👤 Mi chiamo Alice")
print("🤖", chat_sliding_window("Mi chiamo Alice"))
print("👤 Il mio colore preferito è blu")
print("🤖", chat_sliding_window("Il mio colore preferito è blu"))
print("👤 Il mio animale preferito è il gatto")
print("🤖", chat_sliding_window("Il mio animale preferito è il gatto"))
print("👤 Vivo a Roma")
print("🤖", chat_sliding_window("Vivo a Roma"))
print("👤 Come mi chiamo?")
print("🤖", chat_sliding_window("Come mi chiamo?"))

👤 Mi chiamo Alice
🤖 Piacere di conoscerti, Alice! 😊

Come posso aiutarti oggi?
👤 Il mio colore preferito è blu
🤖 Che bello! 💙 Il blu è un colore bellissimo - trasmette calma e serenità. 

C'è un particolare tipo di blu che preferisci? Ad esempio, un blu cielo, un blu marino, o un blu più scuro?
👤 Il mio animale preferito è il gatto
🤖 I gatti sono fantastici! 🐱 Sono intelligenti, affettuosi e hanno un carattere davvero affascinante.

Hai un gatto? Se sì, come si chiama e qual è la sua personalità?
👤 Vivo a Roma
🤖 Che città meravigliosa! 🏛️ Roma è straordinaria con la sua storia millenaria, i monumenti incredibili e l'atmosfera unica.

Ci vivi da sempre o ti sei trasferita lì? Quali sono i tuoi posti preferiti a Roma?
👤 Come mi chiamo?
🤖 Ti chiami **Alice**! 😊

Lo hai detto all'inizio della nostra conversazione. So anche che ami il blu, i gatti e vivi a Roma - sto imparando tante cose interessanti su di te!


### Esercizio 3 — Chatbot con streaming ★★☆
Riscrivi la funzione `chat()` usando lo streaming. L'output deve apparire parola per parola nel terminale. Aggiungi anche il conteggio dei token al termine.

In [13]:
# ESERCIZIO 3
def chat_streaming(messaggio, history, system=None):
 history.append({"role": "user", "content": messaggio})

 # TODO: usa client.messages.stream() invece di client.messages.create()
 # Stampa ogni token con print(text, end='', flush=True)
 # Accumula il testo completo in una variabile
 # Aggiungi la risposta completa alla history
 # Alla fine stampa il numero di token usati
 params = {
  "model": "claude-haiku-4-5-20251001",
  "max_tokens": 300,
  "messages": history
 }
 if system:
  params["system"] = system

 full_text = ""
 with client.messages.stream(**params) as stream:
  for text in stream.text_stream:
   print(text, end="", flush=True)
   full_text += text

 print(f"\n\n📊 Token usati: {len(full_text.split())}")
 history.append({"role": "assistant", "content": full_text})
 return full_text

# Test
history = []
chat_streaming("Spiegami RAG in 3 frasi", history)

# RAG (Retrieval-Augmented Generation) in 3 frasi

1. **RAG è una tecnica che combina** un sistema di ricerca (che trova documenti rilevanti) con un modello di intelligenza artificiale generativa (che crea risposte).

2. **Funziona così**: prima recupera informazioni pertinenti da una base di dati o documenti, poi le passa al modello AI per generare una risposta più accurata e fondata su dati reali.

3. **Il vantaggio principale** è che le risposte dell'AI sono più affidabili, aggiornate e basate su fonti concrete, evitando allucinazioni e informazioni inventate.

📊 Token usati: 86


"# RAG (Retrieval-Augmented Generation) in 3 frasi\n\n1. **RAG è una tecnica che combina** un sistema di ricerca (che trova documenti rilevanti) con un modello di intelligenza artificiale generativa (che crea risposte).\n\n2. **Funziona così**: prima recupera informazioni pertinenti da una base di dati o documenti, poi le passa al modello AI per generare una risposta più accurata e fondata su dati reali.\n\n3. **Il vantaggio principale** è che le risposte dell'AI sono più affidabili, aggiornate e basate su fonti concrete, evitando allucinazioni e informazioni inventate."

### Esercizio 4 — Chatbot con memoria persistente ★★★ (Deliverable!)

Costruisci il chatbot completo `chatbot_cli.py` con:
- History multi-turno
- Sliding window (max 10 messaggi)
- Streaming
- Memoria su JSON persistente
- System prompt WiData
- Loop interattivo con `input()` (digita 'esci' per uscire)

In [19]:
# ESERCIZIO 4 — Chatbot completo (DELIVERABLE)
# Scrivi qui il codice completo del chatbot

import anthropic, json, os
from google.colab import userdata

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
client = anthropic.Anthropic()

MEMORY_FILE = "chatbot_widata.json"
MAX_MESSAGGI = 10

SYSTEM = """
Sei un assistente di supporto clienti WiData.
Rispondi in modo professionale, chiaro e coerente.
Parla solo di prodotti e soluzioni IoT WiData.
Per i prezzi rimanda al commerciale.
Se la domanda è fuori tema, rifiuta cortesemente.
"""

def carica_storia():
    # ← implementa
    if os.path.exists(MEMORY_FILE):
        with open(MEMORY_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    return []

def salva_storia(history):
    # ← implementa
    with open(MEMORY_FILE, "w", encoding="utf-8") as f:
        json.dump(history, f, ensure_ascii=False, indent=2)

def chat(messaggio, history):
    # ← implementa con streaming + sliding window
    history.append({"role": "user", "content": messaggio})

    if len(history) > MAX_MESSAGGI:
        history[:] = history[-MAX_MESSAGGI:]

    params = {
        "model": "claude-haiku-4-5-20251001",
        "max_tokens": 400,
        "messages": history,
        "system": SYSTEM
    }

    full_text = ""
    with client.messages.stream(**params) as stream:
        for text in stream.text_stream:
            print(text, end="", flush=True)
            full_text += text

    print()
    history.append({"role": "assistant", "content": full_text})

    if len(history) > MAX_MESSAGGI:
        history[:] = history[-MAX_MESSAGGI:]

    salva_storia(history)
    return full_text

# Loop principale
def main():
    history = carica_storia()
    print("🤖 Chatbot WiData avviato. Digita 'esci' per uscire.\n")

    while True:
        utente = input("Tu: ")
        if utente.lower() == "esci":
            print("👋 Arrivederci!")
            break
        print("WiData:", end=" ")
        risposta = chat(utente, history)
        # Lo streaming stampa già durante l'esecuzione

main()

🤖 Chatbot WiData avviato. Digita 'esci' per uscire.

Tu: Ciao
WiData: Buongiorno! 👋

Benvenuto in WiData! Sono qui per aiutarti con informazioni sui nostri **prodotti e soluzioni IoT**.

Come posso assisterti oggi? Puoi chiedermi di:

- **Piattaforme IoT** e loro funzionalità
- **Sensori e dispositivi** WiData
- **Integrazioni** e connettività
- **Casi d'uso** e applicazioni
- Qualsiasi domanda tecnica sui nostri servizi

Dimmi pure, come posso esserti utile? 😊
Tu: esci
👋 Arrivederci!


---
## 📤 Consegna

1. Completa tutti gli esercizi
2. Scarica: `File → Scarica → .ipynb`
3. Rinomina: `Lezione3_TUONOME.ipynb`
4. Carica su GitHub in `lezione3/`

```bash
git add lezione3/
git commit -m "Lezione 3 completata"
git push
```

---
### 📖 Per la prossima lezione (Giovedì 28/05)
Leggi **Huyen Cap. 6 — sezione RAG**

---
*ITS Novitas 4.0 — AI Engineering Fundamentals | Marco Uras*